## Estructura Inicial para la Descarga de los Archivos de rtve

In [1]:
import time
import json
import os
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

In [2]:
# ══════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════
URL_BUSCADOR = "https://23fbuscador.rtve.es/"
OUTPUT_DIR   = "documentos_23f"
DELAY        = 2
HEADLESS     = True   # Se cambia a False si quiero ver el navegador "haciendo el trabajo"
# ══════════════════════════════════════════════════════

def crear_driver():
    options = Options()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

# ══════════════════════════════════════════════════════
# CAMBIAR EL PAGINADO A 200 QUE ES EL MAXIMO
# ══════════════════════════════════════════════════════

def seleccionar_200_por_pagina(driver):
    """
    Selecciona la opción '200 / página' para cargar todos los documentos
    en una sola página y no tener que paginar.
    """
    try:
        wait = WebDriverWait(driver, 15)
        # Primer interto con CSS
        selectores_css = [
            "select[name='page-size-select']",
            "select.page-size",
            "select.pagination-size",
        ]
        select_elem = None
        for sel in selectores_css:
            try:
                select_elem = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, sel)))
                break
            except TimeoutException:
                continue
        # Segundo intenfo por XPath si no funciona por CSS
        if select_elem is None:
            select_elem = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//select[option[contains(text(),'200')]]")
            ))

        select = Select(select_elem)
        # Guardar texto previo a hacer clic para evitar StaleElementReferenceException
        texto_opcion = next((o.text for o in select.options if "200" in o.text), None)
        if texto_opcion:
            select.select_by_visible_text(texto_opcion)
            print(f"✔ Seleccionada opción: '{texto_opcion}'")
            time.sleep(3)
            return True

    except (TimeoutException, NoSuchElementException) as e:
        print(f"⚠ No se pudo seleccionar 200/página: {e}")

    return False

# ══════════════════════════════════════════════════════
# EXTRAER LA TABLA CON EL LISTADO DE LOS DOCUMENTOS DISPONIBLES
# ══════════════════════════════════════════════════════

def extraer_tabla(driver):
    """
    Extrae todas las filas de la tabla de documentos.
    """
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//table//tbody//tr"))
        )
    except TimeoutException:
        print("⚠ No apareció la tabla")
        return []

    filas = driver.find_elements(By.XPATH, "//table//tbody//tr")
    print(f"  → {len(filas)} filas en la tabla")

    # Cabeceras
    try:
        ths = driver.find_elements(By.XPATH, "//table//thead//th")
        cabeceras = [th.text.strip() for th in ths]
        print(f"  ℹ Columnas: {cabeceras}")
    except Exception:
        cabeceras = []

    documentos = []
    for i, fila in enumerate(filas):
        celdas = fila.find_elements(By.TAG_NAME, "td")
        doc = {"fila_num": i + 1}

        for j, celda in enumerate(celdas):
            col_name = cabeceras[j] if j < len(cabeceras) else f"col_{j+1}"
            col_name = col_name.lower().replace(" ", "_") or f"col_{j+1}"

            try:
                a = celda.find_element(By.TAG_NAME, "a")
                doc["url"]    = a.get_attribute("href")
                doc[col_name] = a.text.strip()
            except NoSuchElementException:
                doc[col_name] = celda.text.strip()

        documentos.append(doc)

    return documentos


def extraer_transcripcion(driver, url):

    driver.get(url)
    time.sleep(DELAY)

    # XPath confirmado a traves de la revision de la pagina https://23fbuscador.rtve.es/
    # uso del // para generalizar
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//main//section[3]//article//pre"))
        )
        elementos = driver.find_elements(By.XPATH, "//main//section[3]//article//pre")
        textos = [e.text.strip() for e in elementos if e.text.strip()]
        if textos:
            return "\n\n".join(textos)
    except TimeoutException:
        pass

    # Fallback 1: cualquier <pre> en main
    try:
        elementos = driver.find_elements(By.XPATH, "//main//pre")
        textos = [e.text.strip() for e in elementos if e.text.strip()]
        if textos:
            print("    ⚠ fallback: //main//pre")
            return "\n\n".join(textos)
    except Exception:
        pass

    # Fallback 2: todo el <main>
    try:
        print("    ⚠ fallback: texto completo de <main>")
        return driver.find_element(By.TAG_NAME, "main").text.strip()
    except Exception:
        return ""

def nombre_seguro(texto, fallback):
    nombre = re.sub(r'[\\/*?:"<>|]', "", texto or fallback)
    nombre = nombre.replace(" ", "_").strip("._")
    return nombre[:80] or fallback

def guardar(doc, transcripcion):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    titulo = next(
        (v for k, v in doc.items() if k not in ("fila_num", "url") and v),
        f"doc_{doc['fila_num']}"
    )
    base = nombre_seguro(titulo, f"doc_{doc['fila_num']}")

    # .txt legible
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════
# Se deja comentado la inclusion del URL, titulo, numero de paginas, resumen y palabras claves en el archivo TXT
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════
    with open(os.path.join(OUTPUT_DIR, f"{base}.txt"), "w", encoding="utf-8") as f:
#        for k, v in doc.items():
#            if k != "fila_num" and v:
#                f.write(f"{k.upper():15}: {v}\n")
#        f.write("\n" + "═" * 60 + "\n\n")
        f.write(transcripcion)
# ══════════════════════════════════════════════════════
# Se incluye la generacion del json para revision posterior
# ══════════════════════════════════════════════════════
    # .json con todo
    with open(os.path.join(OUTPUT_DIR, f"{base}.json"), "w", encoding="utf-8") as f:
        json.dump({**doc, "transcripcion": transcripcion}, f, ensure_ascii=False, indent=2)

    return base, len(transcripcion)

In [3]:
# ══════════════════════════════════════════════════════
# CREACION PARA LA EJECUCION
# ══════════════════════════════════════════════════════

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    driver = crear_driver()
    todos  = []

    print(f"\n{'═'*55}")
    print(f"  Scraper 23F - Documentos desclasificados RTVE")
    print(f"{'═'*55}\n")

    print("🔍 Abriendo buscador en 23fbuscador.rtve.es ...")
    driver.get(URL_BUSCADOR)
    time.sleep(4)

    print("⚙  Seleccionando 200 documentos por página...")
    seleccionar_200_por_pagina(driver)

    print("\n📋 Extrayendo tabla de documentos...")
    documentos = extraer_tabla(driver)

    if not documentos:
        print("✘ No se encontraron documentos. Prueba con HEADLESS=False para ver qué ocurre.")
        driver.quit()
        return

    print(f"\n🚀 Descargando transcripciones de {len(documentos)} documentos...\n")

    for doc in documentos:
        url = doc.get("url", "")
        titulo = next(
            (v for k, v in doc.items() if k not in ("fila_num", "url") and v),
            f"doc_{doc['fila_num']}"
        )
        titulo_corto = titulo[:55] + ("..." if len(titulo) > 55 else "")
        print(f"  [{doc['fila_num']:>3}/{len(documentos)}] {titulo_corto}")

        if not url:
            print("         ⚠ Sin URL, saltando")
            todos.append(doc)
            continue

        transcripcion = extraer_transcripcion(driver, url)
        base, chars = guardar(doc, transcripcion)
        doc["archivo"] = f"{base}.txt"
        todos.append(doc)
        print(f"         ✔ {base}.txt ({chars:,} caracteres)")

        # Volver al buscador para la siguiente iteración
        driver.get(URL_BUSCADOR)
        time.sleep(DELAY)

    # Índice global
    indice_path = os.path.join(OUTPUT_DIR, "_indice.json")
    with open(indice_path, "w", encoding="utf-8") as f:
        json.dump(todos, f, ensure_ascii=False, indent=2)

    print(f"\n{'═'*55}")
    print(f"  📊 Total documentos : {len(todos)}")
    print(f"  📁 Carpeta          : ./{OUTPUT_DIR}/")
    print(f"  📋 Índice           : {indice_path}")
    print(f"{'═'*55}\n")

    driver.quit()


if __name__ == "__main__":
    main()



═══════════════════════════════════════════════════════
  Scraper 23F - Documentos desclasificados RTVE
═══════════════════════════════════════════════════════

🔍 Abriendo buscador en 23fbuscador.rtve.es ...
⚙  Seleccionando 200 documentos por página...
✔ Seleccionada opción: '200 / página'

📋 Extrayendo tabla de documentos...
  → 167 filas en la tabla
  ℹ Columnas: ['Título', 'Páginas', '', 'Resumen', 'Palabras clave']

🚀 Descargando transcripciones de 167 documentos...

  [  1/167] Vista oral 2/81 del Consejo Supremo de Justicia Militar...
         ✔ Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(20_de_febrero_de_1982).txt (3,934 caracteres)
  [  2/167] Vista oral 2/81 del Consejo Supremo de Justicia Militar...
         ✔ Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(22_de_febrero_de_1982).txt (6,417 caracteres)
  [  3/167] Vista oral 2/81 del Consejo Supremo de Justicia Militar...
         ✔ Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(24_de_febrero_de

In [4]:
# ══════════════════════════════════════════════════════
# REVISION DE UN DOCUMENTO PARA VALIDAR INFORMACION
# ══════════════════════════════════════════════════════
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

URL_DOCUMENTO = "https://23fbuscador.rtve.es/document/ocr/1846?page_size=200&page=1"

driver.get(URL_DOCUMENTO)
time.sleep(3)

print("=" * 60)
print("TÍTULO DE LA PÁGINA:", driver.title)
print("=" * 60)

# Ver todas las secciones
secciones = driver.find_elements(By.XPATH, "//main//section")
print(f"\n{len(secciones)} sección(es) en <main>:")
for i, s in enumerate(secciones, 1):
    texto = s.text.strip()[:120].replace("\n", " ")
    print(f"  section[{i}]: {texto}...")

# Ver todos los <pre>
print("\n--- Elementos <pre> ---")
pres = driver.find_elements(By.TAG_NAME, "pre")
for i, p in enumerate(pres, 1):
    print(f"  pre[{i}]: {p.text[:200]}...")

# Ver todos los <h1>, <h2>, <h3>
print("\n--- Títulos ---")
for tag in ["h1", "h2", "h3"]:
    elems = driver.find_elements(By.TAG_NAME, tag)
    for e in elems:
        if e.text.strip():
            print(f"  <{tag}>: {e.text.strip()}")

# Ver todos los <table>
print("\n--- Tablas ---")
tablas = driver.find_elements(By.TAG_NAME, "table")
print(f"  {len(tablas)} tabla(s)")
for i, t in enumerate(tablas, 1):
    print(f"  tabla[{i}]: {t.text[:200]}...")

# Ver metadatos visibles
print("\n--- Todo el texto del <main> ---")
try:
    main = driver.find_element(By.TAG_NAME, "main")
    print(main.text[:1000])
except:
    pass

driver.quit()

TÍTULO DE LA PÁGINA: Detalle documento

3 sección(es) en <main>:
  section[1]: RESUMEN El documento corresponde al desarrollo de la vista oral 2/81 del Consejo Supremo de Justicia Militar, celebrada ...
  section[2]: PALABRAS CLAVE 10 N/Ref. C/SG/4346/16-03-82 VISTA ORAL 2/81 CONSEJO SUPREMO DE JUSTICIA MILITAR 16-03-82 Fiscal Coronel ...
  section[3]: TEXTO COMPLETO N/Ref. C/SG/4346/16-03-82  # NOTA INFORMATIVA  ASUNTO: VISTA ORAL 2/81 DEL CONSEJO SUPREMO DE JUSTICIA MI...

--- Elementos <pre> ---
  pre[1]: N/Ref. C/SG/4346/16-03-82

# NOTA INFORMATIVA

ASUNTO: VISTA ORAL 2/81 DEL CONSEJO SUPREMO DE JUSTICIA MILITAR.

## 1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 16-03-82.

Comenzó la sesión a las 10...

--- Títulos ---
  <h2>: Vista oral 2/81 del Consejo Supremo de Justicia Militar (16 de marzo de 1982).
  <h3>: RESUMEN
  <h3>: TEXTO COMPLETO
  <h3>: ORIGINAL DIGITALIZADO / AUDIO / VIDEO

--- Tablas ---
  0 tabla(s)

--- Todo el texto del <main> ---
VOLVER AL LISTADO
ANTERIOR
Doc

In [5]:
driver = crear_driver()
url = "https://23fbuscador.rtve.es/document/ocr/1846?page_size=200&page=1"
transcripcion = extraer_transcripcion(driver, url)


In [6]:
transcripcion


'N/Ref. C/SG/4346/16-03-82\n\n# NOTA INFORMATIVA\n\nASUNTO: VISTA ORAL 2/81 DEL CONSEJO SUPREMO DE JUSTICIA MILITAR.\n\n## 1.- DESARROLLO DE LA SESION CORRESPONDIENTE AL 16-03-82.\n\nComenzó la sesión a las 10 horas, continuando el Fiscal con el interrogatorio del Coronel Ibáñez Inglés, interrumpido el día anterior, que tuvo una duración de 1 hora y 45 minutos.\n\nA continuación se procedió a la intervención de los abogados defensores, comenzando por el suyo propio, Sr. Escandell, y continuando con el resto hasta la terminación de la sesión, a las 13,50; en estas intervenciones renunciaron a su turno de preguntas cinco letrados, no preguntando nada tampoco ningún miembro del Consejo.\n\nEl Sr. Escandell pidió la palabra para solicitar, de nuevo, un careo por considerar que había muchas dudas, y que le amparaba el artículo 753 del C.J.M. El Fiscal intervino para opinar que no debía haberlo, así como que no procedía interponer recurso de casación. Le apoyó el Sr. Hermosilla, y el Preside

In [7]:
# encuentra_ids.py
import requests

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# La URL del listado completo con 200 por página
url = "https://23fbuscador.rtve.es/?q=&page_size=200&q_all=&q_any=&q_not="

r = requests.get(url, headers=headers)
print("Status:", r.status_code)

# Buscar todos los IDs de documentos en el HTML
import re
ids = re.findall(r'/document/ocr/(\d+)', r.text)
ids_unicos = sorted(set(int(i) for i in ids))

print(f"\nIDs encontrados: {len(ids_unicos)}")
print(f"Rango: {min(ids_unicos)} → {max(ids_unicos)}")
print(f"\nLista completa: {ids_unicos}")

Status: 200

IDs encontrados: 167
Rango: 1694 → 1860

Lista completa: [1694, 1695, 1696, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1705, 1706, 1707, 1708, 1709, 1710, 1711, 1712, 1713, 1714, 1715, 1716, 1717, 1718, 1719, 1720, 1721, 1722, 1723, 1724, 1725, 1726, 1727, 1728, 1729, 1730, 1731, 1732, 1733, 1734, 1735, 1736, 1737, 1738, 1739, 1740, 1741, 1742, 1743, 1744, 1745, 1746, 1747, 1748, 1749, 1750, 1751, 1752, 1753, 1754, 1755, 1756, 1757, 1758, 1759, 1760, 1761, 1762, 1763, 1764, 1765, 1766, 1767, 1768, 1769, 1770, 1771, 1772, 1773, 1774, 1775, 1776, 1777, 1778, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1802, 1803, 1804, 1805, 1806, 1807, 1808, 1809, 1810, 1811, 1812, 1813, 1814, 1815, 1816, 1817, 1818, 1819, 1820, 1821, 1822, 1823, 1824, 1825, 1826, 1827, 1828, 1829, 1830, 1831, 1832, 1833, 1834, 1835, 1836, 1837, 1838, 1839, 1840, 1841, 1842, 1843, 1844, 1845, 1846, 1847, 1848,